In [29]:
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.metrics import (homogeneity_score,v_measure_score,adjusted_mutual_info_score,normalized_mutual_info_score,adjusted_rand_score,fowlkes_mallows_score)


In [30]:
repeat_times = 2
simple_path = '/home/cavin/jt/benchmark/data/hbc/s1_filtered_cells.h5ad'
celltype_col = 'cell_type'
cell_emb_col = 'X_pca'
pca_components = 50
random_seed=2026 + repeat_times * 200
save_path = "/home/cavin/jt/benchmark/experiments/embedings/batch_integrate/PCA_human_breast_cancer_emb.parquet"

In [31]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [32]:
adata.obsm[cell_emb_col] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial', 'X_pca'

In [33]:
res_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
key_added='leiden'
sc.pp.neighbors(adata, use_rep=cell_emb_col,random_state=random_seed)
adata


AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    uns: 'neighbors'
    obsm: 'spatial', 'X_pca'
    obsp: 'distances', 'connectivities'

In [34]:
max_value = 0
metrics = {"method": "PCA", "ARI": 0, "NMI": 0, "AMI": 0, "HS": 0, "VM": 0, "FMI": 0, "mean value": 0}
save_label_df = None

In [35]:

for used_res in res_list:
    sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=key_added)
    true_labels = np.array(adata.obs[celltype_col])
    cluster_labels = np.array(adata.obs[key_added])
    FMI = fowlkes_mallows_score(true_labels, cluster_labels)
    homogeneity = homogeneity_score(true_labels, cluster_labels)
    v_measure = v_measure_score(true_labels, cluster_labels)
    ami = adjusted_mutual_info_score(true_labels, cluster_labels)
    nmi = normalized_mutual_info_score(true_labels, cluster_labels)
    ari = adjusted_rand_score(true_labels, cluster_labels)
    mean_value = np.mean([FMI, homogeneity, v_measure, ami, nmi, ari])

    n_cluster = len(adata.obs[key_added].unique())
    n_celltype = len(adata.obs[celltype_col].unique())
    if mean_value > max_value:
        save_label_df = adata.obs[key_added]
        metrics["ARI"] = ari
        metrics["NMI"] = nmi
        metrics["AMI"] = ami
        metrics["HS"] = homogeneity
        metrics["VM"] = v_measure
        metrics["FMI"] = FMI
        metrics["mean value"] = mean_value
        max_value = mean_value

    print(f'resolution = {used_res} | n_celltype = {n_celltype} | n_cluster = {n_cluster} | mean_value = {round(mean_value,3)} | ARI = {round(ari,3)} |  NMI = {round(nmi,3)} | AMI = {round(ami,3)} | HS = {round(homogeneity,3)} | VM = {round(v_measure,3)} | FMI = {round(FMI,3)} ')

resolution = 0.1 | n_celltype = 20 | n_cluster = 4 | mean_value = 0.408 | ARI = 0.282 |  NMI = 0.447 | AMI = 0.447 | HS = 0.314 | VM = 0.447 | FMI = 0.511 
resolution = 0.2 | n_celltype = 20 | n_cluster = 7 | mean_value = 0.546 | ARI = 0.478 |  NMI = 0.574 | AMI = 0.574 | HS = 0.507 | VM = 0.574 | FMI = 0.566 
resolution = 0.3 | n_celltype = 20 | n_cluster = 9 | mean_value = 0.595 | ARI = 0.536 |  NMI = 0.619 | AMI = 0.619 | HS = 0.577 | VM = 0.619 | FMI = 0.603 
resolution = 0.4 | n_celltype = 20 | n_cluster = 10 | mean_value = 0.59 | ARI = 0.521 |  NMI = 0.616 | AMI = 0.616 | HS = 0.579 | VM = 0.616 | FMI = 0.589 
resolution = 0.5 | n_celltype = 20 | n_cluster = 11 | mean_value = 0.547 | ARI = 0.414 |  NMI = 0.599 | AMI = 0.599 | HS = 0.581 | VM = 0.599 | FMI = 0.492 
resolution = 0.6 | n_celltype = 20 | n_cluster = 11 | mean_value = 0.544 | ARI = 0.405 |  NMI = 0.599 | AMI = 0.599 | HS = 0.582 | VM = 0.599 | FMI = 0.483 
resolution = 0.7 | n_celltype = 20 | n_cluster = 12 | mean_val

In [36]:
metrics

{'method': 'PCA',
 'ARI': 0.5356949356853935,
 'NMI': 0.6186443366892899,
 'AMI': 0.6185970314245638,
 'HS': 0.576718783126803,
 'VM': 0.6186443366892899,
 'FMI': 0.6034253860864901,
 'mean value': 0.5952874682836383}

In [37]:
save_label_df

s1r1_1         4
s1r1_2         4
s1r1_5         4
s1r1_8         4
s1r1_9         4
              ..
s1r2_118748    5
s1r2_118749    5
s1r2_118750    5
s1r2_118751    5
s1r2_118752    5
Name: leiden, Length: 281780, dtype: category
Categories (9, object): ['0', '1', '2', '3', ..., '5', '6', '7', '8']

In [38]:
save_label_df_path = f"/home/cavin/jt/benchmark/experiments/results/labels_df/breast_cancer/pca_human_breast_cancer_labels_repeat_{repeat_times}.csv"

In [39]:
save_label_df.to_csv(save_label_df_path)

In [40]:
metrics_df = pd.DataFrame.from_dict(metrics, orient='index').T

In [41]:
metrics_df_save_path = f"/home/cavin/jt/benchmark/experiments/results/cluster_metrics/breast_cancer/human_breast_cancer_metrics_repeat_{repeat_times}.csv"

In [42]:
metrics_df.to_csv(metrics_df_save_path, index=False,mode="a", header=not pd.io.common.file_exists(metrics_df_save_path))